Estudiante: SANCAN CHOEZ JAVIER

Construir una red neuronal multicapa completa que clasifique las tres especies del dataset Iris. Incluir: normalización de datos, ReLU en capas ocultas, softmax en la salida, entrenamiento con cross-entropy y evaluación con accuracy. Y esta debe contener la función def predecir(self, X): return np.argmax(self.forward(X), axis=1)

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
#Funciones de Activacion y Utilidades
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x > 0).astype(float)

def softmax(x):
    #Restamos el max para estabilidad numérica y evitar overflow
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [3]:
class MLP_Iris:
    def __init__(self, n_capas, lr=0.01):
        self.lr = lr
        self.W = []
        self.b = []
        #Arquitectura Inicialización para ReLU
        for i in range(len(n_capas) - 1):
            w = np.random.randn(n_capas[i], n_capas[i+1]) * np.sqrt(2.0 / n_capas[i])
            b = np.zeros((1, n_capas[i+1]))
            self.W.append(w)
            self.b.append(b)
        
    def forward(self, X):
        self.A = [X]
        self.Z = []
        activa = X
        for i in range(len(self.W)):
            z = np.dot(activa, self.W[i]) + self.b[i]
            #Decisión ReLU en capas ocultas, Softmax en la última
            a = softmax(z) if i == len(self.W) - 1 else relu(z)
            self.Z.append(z)
            self.A.append(a)
            activa = a
        return activa

    def backward(self, y_real):
        m = y_real.shape[0]
        #Gradiente simplificado para Cross-Entropy + Softmax
        delta = (self.A[-1] - y_real) 

        for i in range(len(self.W) - 1, -1, -1):
            dw = np.dot(self.A[i].T, delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m
            
            if i > 0:
                #Propagacion del error usando la derivada de ReLU
                delta = np.dot(delta, self.W[i].T) * relu_derivada(self.Z[i-1])
            
            self.W[i] -= self.lr * dw
            self.b[i] -= self.lr * db

    def entrenamiento(self, X, y, epocas=1000):
        for ep in range(epocas):
            salida = self.forward(X)
            self.backward(y)
            if ep % 200 == 0:
                #Loss Categorical Cross-Entropy
                loss = -np.mean(np.sum(y * np.log(salida + 1e-8), axis=1))
                print(f"Epoca {ep} - Loss: {loss:.4f}")

    #Función de predicción
    def predecir(self, X):
        return np.argmax(self.forward(X), axis=1)

In [4]:
#Preparacion Dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

In [5]:
#Normalización - Escalar datos entre 0 y 1 para estabilidad
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

In [6]:
#One-Hot Encoding para la salida Softmax
y_one_hot = np.eye(3)[y]

In [7]:
#Entrenamiento y prueba dividido
X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, test_size=0.2, random_state=42)

In [8]:
#Ejecucion Arquitectura: 4 entradas, 8 neuronas ocultas, 3 salidas
red = MLP_Iris(n_capas=[4, 8, 3], lr=0.1) # Learning Rate 0.1 para un aprendizaje rapido pero estable
red.entrenamiento(X_train, y_train, epocas=1000) #Epocas=1000 iteraciones para asegurar que el gradiente encuentre el mínimo

Epoca 0 - Loss: 1.7527
Epoca 200 - Loss: 0.3337
Epoca 400 - Loss: 0.1857
Epoca 600 - Loss: 0.1286
Epoca 800 - Loss: 0.1020


In [9]:
#Resultados con Accuracy
predicciones = red.predecir(X_test)
y_test_labels = np.argmax(y_test, axis=1)
accuracy = np.mean(predicciones == y_test_labels)

print(f"\nAccuracy Final en Test: {accuracy * 100:.2f}%")


Accuracy Final en Test: 96.67%
